# Security, Secrets & Hardening

## What's covered

- **The container threat model** — what container isolation protects against, and what it explicitly doesn't
- **The kernel primitives** revisited — namespaces, cgroups, capabilities, seccomp, AppArmor/SELinux
- **The "root in container" problem** — why running as root is almost never necessary, and what it lets an attacker do
- **Running as non-root** — `USER` in Dockerfile, `--user` at run time, fixing file ownership
- **User-namespace remapping** — daemon-level UID translation, `userns-remap`
- **Rootless Docker** — running the daemon itself unprivileged
- **Linux capabilities** — `--cap-drop=ALL` then `--cap-add`, the standard hardened baseline
- **Seccomp profiles** — restricting which syscalls a container can make
- **AppArmor / SELinux** — MAC layers, the default `docker-default` profile
- **Read-only root filesystem** — `--read-only` with `tmpfs` overlays
- **`no-new-privileges`** — blocking setuid escalation
- **Secrets** — why environment variables aren't secret, and what to use instead
- **Compose, Swarm, and BuildKit secrets** — the three places "real" secrets live in the Docker ecosystem
- **Image signing** — `cosign`, `docker trust`, in-toto attestations
- **Supply-chain hygiene** — SBOMs, provenance attestations, the path from `docker build` to a signed, scanned, provenanced artifact

By the end you should be able to write a hardened Dockerfile, run a container with the surface area an auditor will approve of, and explain to a teammate why `-e SECRET=...` is a bug, not a feature.

## The container threat model

Before stacking defences, be clear about what containers *are* and aren't.

**Containers are a process-isolation mechanism, not a security boundary.** They keep workloads from accidentally tripping over each other on a shared kernel. They are **not** designed to safely host untrusted, hostile code on a multi-tenant kernel — that's what virtual machines (and projects like Kata Containers, gVisor) are for.

What container isolation protects against by default:

- A container reading or modifying another container's files (each has its own mount namespace).
- A container seeing another's processes (each has its own PID namespace).
- A container hogging all the CPU/RAM (cgroups limit per-container resources).
- A container's process binding the host's privileged ports (network namespace, plus dropping the `NET_BIND_SERVICE` cap if you want).

What it **does not** protect against, by default:

- A container running as root that breaks out via a kernel vulnerability (`CVE-2022-0185`, `CVE-2024-1086`, etc).
- A container with `--privileged` doing essentially anything to the host.
- A container that gets `CAP_SYS_ADMIN` mounting host filesystems or escaping via `cgroups`.
- A container that mounts `/var/run/docker.sock` — that's a remote control of the daemon, which means root on the host.
- A malicious or compromised image (supply-chain attacks).

Hardening = adding layers of constraint until the surface area is small enough that a compromise in the container is unlikely to compromise the host.

## The kernel primitives revisited

From notebook 01, Docker composes Linux kernel features. Quick reminder of what each one buys you:

| Primitive | What it does | Notes |
|---|---|---|
| **Namespaces** | Isolate views of the system (PID, mount, net, IPC, UTS, user, cgroup) | What makes a "container" a container |
| **cgroups** | Account and limit resources (CPU, memory, I/O, PIDs) | Notebook 03 — the `--memory`/`--cpus` flags |
| **Capabilities** | Slice the all-powerful root into ~40 individual privileges | `CAP_NET_BIND_SERVICE`, `CAP_SYS_ADMIN`, etc |
| **seccomp** | Allow- or block-list specific syscalls | Default profile blocks ~50 dangerous syscalls |
| **AppArmor** / **SELinux** | Mandatory access control on top of file permissions | Distro-dependent. Docker ships a `docker-default` AppArmor profile |
| **User namespaces** | Map UIDs inside the container to different UIDs on the host | Root in container can be UID 100000 on the host |

The default config Docker gives a container is already much tighter than running the same process directly on the host:

- 40+ capabilities dropped (only ~14 retained by default)
- ~50 syscalls blocked by the default seccomp profile
- AppArmor's `docker-default` profile applied on systems with AppArmor

But "much tighter than nothing" isn't the same as "tight enough for production." The rest of this notebook is "how much further you should crank each dial."

## The "root in container" problem

By default the container's main process runs as **root inside its namespace** — UID 0. People often think "well, it's just root inside the container, that doesn't matter." It mostly does.

Root in the container can:

- Modify any file in the container's filesystem (including ones bind-mounted from the host as root).
- Bind to privileged ports inside the container.
- Use any capability the container still has (which is a lot in the default config).
- Use a kernel CVE — and there are several every year — to escape into host context.
- Read its own credentials, including secrets passed in env vars.

Plus, files the container creates on a bind-mounted host directory are **owned by UID 0 on the host** unless you've configured user-namespace remapping. So a "root in container" with a host bind mount is effectively root on those files on the host.

The fix is mundane and effective: **don't run as root.** Most containerised apps don't need to.

## Running as non-root

Two places to set the user.

### In the Dockerfile

```dockerfile
FROM python:3.12-slim
# ... installation steps as root ...
RUN useradd --system --uid 10001 --user-group --no-create-home appuser
USER appuser
CMD ["python", "app.py"]
```

Best practice:

- **Create a dedicated, system user** for the app. Don't reuse `nobody` (`65534`) — distros differ on what `nobody` is supposed to do.
- **Pick a high UID** (e.g. `10001`) that doesn't collide with common host UIDs (1000, 33, 999, etc).
- **`USER` instruction after** all the steps that need root (package installation, file ownership). Switching too early forces awkward `sudo` patterns inside `RUN`.

### At run time

`docker run -u UID[:GID]` overrides the image's `USER`:

```bash
docker run -u 10001:10001 myorg/api
docker run -u "$(id -u):$(id -g)" -v $(pwd):/work alpine sh   # run as your host user
```

This is the standard fix for the bind-mount file-ownership problem from notebook 04: run the container as your host UID, and the files it creates have your host ownership.

### "But the image requires root"

Some images insist on running as root (the official Nginx, Postgres for some operations, anything using port 80/443 without `CAP_NET_BIND_SERVICE`). Two responses:

- **Map the port.** Run Nginx as non-root inside the container listening on 8080; publish with `-p 80:8080`. The privilege is at the host side, not the container.
- **Use a non-root variant.** `nginxinc/nginx-unprivileged`, `bitnami/postgresql`, etc — many official images have non-root cousins.
- **Grant only the specific capability needed.** `--cap-add=NET_BIND_SERVICE` lets a non-root process bind to low ports.

## User-namespace remapping

By default, root inside a container is **the same UID 0** as root on the host. They share the user namespace.

**User-namespace remapping** changes that. The daemon maps container UID 0 to, say, host UID 100000. From the host's perspective, the container's "root" is just a normal unprivileged user. A breakout via a kernel bug still lands you as unprivileged.

Enable in `/etc/docker/daemon.json`:

```json
{
  "userns-remap": "default"
}
```

`"default"` creates a user `dockremap` on the host with a `/etc/subuid` and `/etc/subgid` allocation, and remaps all containers into that range.

Trade-offs:

- **All containers share the same remap range.** If you need per-container isolation, you need rootless Docker (next section) or Kata.
- **Bind mounts get tricky** — files on the host owned by `100000:100000` look like root inside the container. You have to `chown` accordingly.
- **Some features are incompatible** — `--privileged`, certain `--security-opt` options, sharing IPC/PID namespaces with the host.

For dev/CI, leave it off. For production servers running containers from sources you don't fully trust, turn it on.

## Rootless Docker

`userns-remap` still requires the **daemon** to run as root. **Rootless Docker** runs the entire daemon (and all containers) under your unprivileged user account.

```bash
dockerd-rootless-setuptool.sh install   # one-time
systemctl --user start docker
```

After setup, your shell has `DOCKER_HOST=unix://...` pointed at the rootless daemon. `docker run` does what you'd expect, but everything happens as your user. There is no root daemon to compromise; a kernel escape from a container only gives an attacker your own UID's privileges on the host.

Trade-offs:

- **Networking is slower** by default (uses `slirp4netns`, a userspace proxy). Per-container `--network host` is unavailable.
- **`--privileged` doesn't work** (and shouldn't).
- **Cgroup v2 required** for resource limits. Most modern distros are fine.
- **No `iptables` rules** on the host (rootless can't write them) — port publishing works via `slirp4netns` instead.

Rootless is the right answer for shared dev hosts, sandbox environments, and CI runners where you don't want CI step compromise to mean host compromise.

## Linux capabilities — drop everything, add what you need

The default container drops about half of Linux's 40-ish capabilities. The remaining set is still pretty large and includes some that have led to escapes in the past (`CAP_NET_RAW`, `CAP_SETUID`, `CAP_DAC_OVERRIDE`).

The hardened baseline:

```bash
docker run --cap-drop=ALL --cap-add=NET_BIND_SERVICE myorg/api
```

Drop **all** capabilities, then add back exactly the ones the app needs. For most application code: nothing. For things that bind to low ports: `NET_BIND_SERVICE`. For things that need to change file ownership: `CHOWN`.

In Compose:

```yaml
services:
  api:
    image: myorg/api
    cap_drop: [ALL]
    cap_add: [NET_BIND_SERVICE]
```

### Capabilities you should know

| Capability | Used for | Common in containers? |
|---|---|---|
| `NET_BIND_SERVICE` | Binding ports < 1024 | Nginx, Apache when running as non-root |
| `CHOWN` | `chown` outside your own UID | Init scripts that fix file ownership |
| `SETUID` / `SETGID` | Changing UID/GID after start | Long-lived servers that drop privs after binding |
| `KILL` | Sending signals across users | rare |
| `NET_ADMIN` | Configure network interfaces | VPN containers, network appliances |
| `SYS_ADMIN` | The "almost-root" cap. Mount, swap, more | **Avoid.** Almost never needed for app workloads |
| `SYS_PTRACE` | `strace`/`gdb` | Debugging containers, not production |

The grand fail-safe: `--privileged` gives **all** capabilities and drops all the seccomp/AppArmor profiles. Use only when you can articulate why no other combination works (typically: running Docker-in-Docker, low-level hardware access, IDS containers).

## Seccomp profiles

`seccomp` (secure computing mode) is a kernel feature that filters which syscalls a process can make. Docker applies its `default.json` seccomp profile to every container, blocking about 50 syscalls that have a history of CVEs or aren't needed by normal apps (`mount`, `umount2`, `keyctl`, `reboot`, etc).

For most containers the default is fine. To tighten further:

```bash
docker run --security-opt seccomp=./my-profile.json myorg/api
```

You can write a custom profile (JSON, allow-list style) that lets through only the syscalls your app actually uses. Tools like `oci-seccomp-bpf-hook` and `falco` can trace your app under normal load and emit a minimal profile.

To **disable seccomp entirely** (don't):

```bash
docker run --security-opt seccomp=unconfined myorg/api
```

This is what many "it doesn't work, I gave up" posts on Stack Overflow recommend. Resist. If a syscall is blocked, the better fix is usually to fix the app or add it to the profile.

## AppArmor and SELinux

Two **Mandatory Access Control** systems Linux distros ship. They sit on top of regular file permissions and decide which paths, capabilities, and operations a process can perform.

**AppArmor** (Ubuntu, Debian) — Docker ships a `docker-default` profile applied automatically when AppArmor is enabled. It restricts containers from writing to most of `/proc`, mounting filesystems, raw networking, etc.

```bash
docker run --security-opt apparmor=my-custom-profile myorg/api
```

**SELinux** (Red Hat, Fedora, CentOS) — different model, same idea. Type enforcement labels each file and process; only matching labels can interact. Docker sets container labels with `--security-opt label=...`.

You usually don't touch either. They're applied by default; you tune them when an audit asks why a container has more access than it needs. If you don't have a specific reason to override, the defaults are correct.

## Read-only root filesystem

Most applications don't need to write to most of their filesystem at runtime. They write to **specific** paths (logs, caches, uploads). Marking the rest read-only blocks a huge class of post-compromise behaviours (dropping a webshell, modifying binaries).

```bash
docker run --read-only \
  --tmpfs /tmp \
  --tmpfs /var/run \
  -v applogs:/app/logs \
  myorg/api
```

- `--read-only` makes the container's root filesystem read-only.
- `--tmpfs /tmp` gives `/tmp` writable in RAM (most apps assume `/tmp` is writable).
- `--tmpfs /var/run` for things like PID files.
- `-v applogs:/app/logs` is the persistent volume for things you actually want kept.

In Compose:

```yaml
services:
  api:
    image: myorg/api
    read_only: true
    tmpfs:
      - /tmp
      - /var/run
    volumes:
      - applogs:/app/logs
```

This is one of the highest-leverage hardening moves available — a few minutes of audit work for a much smaller attack surface.

## `no-new-privileges`

A subtle Linux feature: if a binary has the setuid bit, executing it normally elevates the process. `no-new-privileges` tells the kernel "ignore setuid bits for this process tree — it can never gain privileges beyond what it started with."

```bash
docker run --security-opt no-new-privileges myorg/api
```

In Compose:

```yaml
services:
  api:
    image: myorg/api
    security_opt:
      - no-new-privileges:true
```

This breaks `sudo`, `mount`, and other setuid utilities inside the container. **Most app containers don't need them**, and breaking them is a positive — an attacker who lands in your container can't suddenly become root via a setuid binary.

Combine with non-root user and capability dropping for the hardened baseline:

```yaml
services:
  api:
    image: myorg/api
    user: "10001:10001"
    read_only: true
    tmpfs: [/tmp, /var/run]
    cap_drop: [ALL]
    cap_add: [NET_BIND_SERVICE]
    security_opt:
      - no-new-privileges:true
```

## Secrets — and why env vars aren't secret

A pattern from every tutorial:

```bash
docker run -e DB_PASSWORD=hunter2 myorg/api
```

Convenient. **Not secret.** That password leaks in at least four places:

1. **`docker inspect`** prints the env block. Anyone with daemon access reads it.
2. **`/proc/<pid>/environ`** on the host exposes it to anyone who can read it (root, often more).
3. **The shell history** on the launcher's machine.
4. **Crash dumps / log statements** that helpfully include "environment at the time of crash."

Plus, env vars are inherited by child processes — every shell-out inside the container sees the password too.

The proper container-native way: **secret files mounted at a known path**, with the file readable only by the app's user. Three places to get them from.

### Docker Compose `secrets:`

```yaml
services:
  api:
    image: myorg/api
    secrets:
      - db_password

secrets:
  db_password:
    file: ./secrets/db_password.txt
```

The file is mounted at `/run/secrets/db_password` inside the container, with restrictive permissions. The app reads from the file path, not an env var.

### Swarm secrets (notebook 09)

Same syntax in a Compose-flavoured stack file, but Swarm encrypts secrets at rest in the raft log and distributes them only to nodes running services that need them.

### BuildKit secret mounts (build-time)

For secrets needed *during the build* (a private package registry token, an SSH key):

```dockerfile
# syntax=docker/dockerfile:1.7
FROM golang:1.23
RUN --mount=type=secret,id=ghtoken \
    git config --global url."https://oauth2:$(cat /run/secrets/ghtoken)@github.com/".insteadOf "https://github.com/" \
 && go mod download
```

```bash
docker build --secret id=ghtoken,src=$HOME/.gh-token -t myapp .
```

The token is mounted at `/run/secrets/ghtoken` only for the duration of that `RUN` step. **It is not committed to a layer.** Compare with `--build-arg` or `ENV` for secrets — both bake the value into image history, where anyone who pulls the image can extract it.

### External secret managers

For prod, often you don't want the secret in any file on the deploy host — you want the container to fetch it at startup from Vault, AWS Secrets Manager, or GCP Secret Manager. A sidecar or init container handles the fetch and writes to a `tmpfs` for the app to read. That's outside Docker's scope but worth knowing exists.

## Image signing

How does the runtime know the `myorg/api:1.2.3` it just pulled was actually built by your CI and not swapped at the registry? **Image signing.**

### Cosign (modern, widely-adopted)

Part of the Sigstore project. Sign with a private key (or with keyless GitHub OIDC):

```bash
# Sign
cosign sign --key cosign.key ghcr.io/myorg/api:1.2.3

# Verify
cosign verify --key cosign.pub ghcr.io/myorg/api:1.2.3
```

Signatures are stored as separate artifacts in the registry (special tag scheme), associated with the image's digest. Kubernetes admission controllers (Kyverno, Connaisseur) can refuse to schedule unsigned images.

### Docker Content Trust (older)

Built into the Docker CLI via the legacy Notary v1 system:

```bash
export DOCKER_CONTENT_TRUST=1
docker push myorg/api:1.2.3   # signs automatically
docker pull myorg/api:1.2.3   # verifies; fails if unsigned
```

Less commonly used today in favour of Cosign. Notary v2 is in active development but not yet ubiquitous.

### What signing buys you

- **Origin guarantees.** You know the image was built by whoever holds the signing key.
- **Integrity.** Re-pushes can't impersonate (signatures don't move automatically).
- **Audit trail.** You can prove which images were promoted through CI by checking signatures.

## Supply-chain hygiene — SBOMs, provenance, attestations

The newest layer in container security: not just "is this image signed?", but "**what's in it, and how was it built?**"

### SBOM (Software Bill of Materials)

A machine-readable list of every package, version, and license in an image. Two common formats: **SPDX** and **CycloneDX**. Generate with `docker sbom`, `syft`, or BuildKit's built-in support:

```bash
docker buildx build --sbom=true -t myorg/api:1.2.3 --push .

# Inspect
docker buildx imagetools inspect myorg/api:1.2.3 --format '{{ json .SBOM }}'
```

The SBOM is attached as a separate artifact, referenced from the manifest. Downstream consumers can scan the SBOM directly without running the image.

### Provenance attestations

A **provenance attestation** records *how* the image was built — the Dockerfile, the build context's git commit, the builder image, the build arguments. Generated by BuildKit with `--provenance=true`. Together with the SBOM, the consumer can answer "this binary came from this Dockerfile, this source commit, this base image" in a verifiable way.

### SLSA — the framework that ties this together

**SLSA** (Supply-chain Levels for Software Artifacts) is a Google-originated framework that defines build maturity levels:

- **L1** — Provenance exists.
- **L2** — Provenance is signed.
- **L3** — Builds run on hardened, isolated builders.
- **L4** — Two-party review and hermetic builds.

GitHub Actions, GitLab CI, and most modern CI systems can issue SLSA L3 attestations automatically. The chain:

```
git commit  ->  CI build with provenance  ->  signed image + signed SBOM in registry
                                                    |
                                                    v
                                            cluster admission controller verifies
                                            signature + SBOM is clean + provenance OK
```

This is what "modern production container security" looks like in 2026. Notebook 10 (the install/troubleshoot/DCA chapter) revisits some of these as exam topics; this notebook is enough to know what each piece does and why it matters.

## Putting it together — the hardened service

A single Compose service combining everything we've covered:

```yaml
services:
  api:
    image: ghcr.io/myorg/api@sha256:abc123...    # pinned by digest
    user: "10001:10001"                          # non-root
    read_only: true                              # rootfs read-only
    tmpfs:
      - /tmp
      - /var/run
    cap_drop: [ALL]                              # no capabilities
    cap_add: [NET_BIND_SERVICE]                  # just bind low ports
    security_opt:
      - no-new-privileges:true                   # no setuid escalation
    secrets:
      - db_password                              # secret file, not env var
    environment:
      DB_PASSWORD_FILE: /run/secrets/db_password # app reads from the file
    networks:
      - backend
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "wget", "-qO-", "http://localhost:8080/health"]
      interval: 30s
      timeout: 3s
      retries: 3
      start_period: 10s

secrets:
  db_password:
    file: ./secrets/db_password.txt

networks:
  backend:
    driver: bridge
```

And on the registry side, the image referenced here is:

- **Built from a multi-stage Dockerfile** (notebook 02) producing a slim, no-toolchain runtime.
- **Pinned by digest** so the deploy is byte-identical.
- **Signed** with cosign — admission controllers can verify origin.
- **Accompanied by an SBOM** — vulnerability scanners can audit packages.
- **Accompanied by a SLSA provenance attestation** — auditors can verify the build chain.

That's the full picture. None of it is mandatory; all of it is achievable in a CI pipeline of a few hundred YAML lines. The DCA exam (notebook 10) tests a subset of these — non-root, capabilities, secrets, signing — and they're the same things real auditors ask about.